In [1]:
%pip install wikipedia


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import wikipedia

wikipedia.set_lang("ru")
text = wikipedia.page("Памятник_Пушкину_(Москва,_Бауманская_улица)").content

with open("article1.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [ ]:
text = wikipedia.page("Список_творческих_работ_Кэри_Гранта").content

with open("article2.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [ ]:
text = wikipedia.page("Кинтанилья,_Хосе_Антонио").content

with open("article3.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [ ]:
text = wikipedia.page("Сборная_Гондураса_по_футболу").content

with open("article4.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [ ]:
text = wikipedia.page("Футбольная_война").content

with open("article5.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [1]:
%pip install sentencepiece


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import csv
from collections import Counter
import sentencepiece as spm

In [3]:
os.chdir("/home/aigul/Desktop/info_retr/hw2")
texts = []
for i in range(1, 6):
    filename = "article" + str(i) + ".txt"
    with open(os.path.join("", filename)) as f:
        texts.append(f.read())

with open("corpus.txt", "w", encoding="utf-8") as f:
        f.write("\n\n".join(texts).lower())

In [4]:
spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="sp_bpe",
    model_type="bpe",
    vocab_size=1000,
    split_digits=True,
    byte_fallback=False,
)
sp_bpe = spm.SentencePieceProcessor(model_file="sp_bpe.model")

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: corpus.txt
  input_format: 
  model_prefix: sp_bpe
  model_type: BPE
  vocab_size: 1000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_

In [5]:
spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="sp_uni",
    model_type="unigram",
    vocab_size=1000,
    split_digits=True,
    byte_fallback=False,
)
sp_uni = spm.SentencePieceProcessor(model_file="sp_uni.model")

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: corpus.txt
  input_format: 
  model_prefix: sp_uni
  model_type: UNIGRAM
  vocab_size: 1000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  different

In [7]:
import re

corpus_text = open("corpus.txt", "r", encoding="utf-8").read().lower()

word_tokens = re.findall(r"[а-яёa-zA-Z]+(?:-[а-яёa-zA-Z]+)*", corpus_text)
bpe_tokens  = sp_bpe.encode(corpus_text, out_type=str)
uni_tokens  = sp_uni.encode(corpus_text, out_type=str)

word_freq = Counter(word_tokens)
bpe_freq  = Counter(bpe_tokens)
uni_freq  = Counter(uni_tokens)

In [9]:
def freq_to_csv(counter, out_csv):
    with open(out_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["token", "count"])
        for token, cnt in counter.most_common():
            writer.writerow([token, cnt])

freq_to_csv(word_freq, "freq_words.csv")
freq_to_csv(bpe_freq,  "freq_bpe.csv")
freq_to_csv(uni_freq,  "freq_unigram.csv")

In [14]:
TOP_N_PRINT = 30

print("\n=== TOP по словам ===")
for tok, cnt in word_freq.most_common(TOP_N_PRINT):
    print(f"{tok}\t{cnt}")

print("\n=== TOP BPE ===")
for tok, cnt in bpe_freq.most_common(TOP_N_PRINT):
    print(f"{tok}\t{cnt}")

print("\n=== TOP Unigram ===")
for tok, cnt in uni_freq.most_common(TOP_N_PRINT):
    print(f"{tok}\t{cnt}")



=== TOP по словам ===
в	347
и	171
на	90
с	90
гондураса	71
года	50
по	46
сальвадора	42
из	34
году	34
к	34
грант	29
был	26
июля	26
а	25
что	23
были	23
гондурас	23
за	21
после	20
не	20
правительство	20
он	19
между	19
территории	19
его	18
от	18
о	17
для	17
июня	16

=== TOP BPE ===
▁в	400
▁	385
,	325
1	296
.	287
9	254
▁и	198
▁с	169
0	168
▁по	141
▁на	138
н	137
ли	136
▁«	136
м	129
▁(	125
й	120
2	118
р	109
л	108
6	107
е	106
-	103
т	101
х	94
ла	93
на	92
»	89
с	84
д	83

=== TOP Unigram ===
▁	1377
е	594
▁в	431
и	424
а	413
,	382
.	302
1	296
о	286
9	254
▁и	197
т	187
я	184
х	179
▁с	178
0	168
у	159
н	156
ы	142
▁на	137
м	136
▁«	136
й	133
(	125
▁по	123
2	118
»	117
в	110
6	107
р	103


In [15]:
# Демонстрация различий на одном абзаце
print("\n=== Пример ===")
demo = "в список творческих работ англо-американского актёра кэри гранта (1904—1986) входят работы в кинематографе, в театре и на радио, которые охватывают 46 лет его актёрской карьеры (с 1920 по 1966 год). за годы кинокарьеры с 1932 по 1966 год он снялся в семидесяти двух полнометражных фильмах. начав свою актёрскую карьеру в 1920-е годы с участия в водевилях и на бродвее, в 1931 году грант перебрался в голливуд, где он начал сниматься в небольших ролях в комедийных фильмах и драмах докодексового периода. в конце 1930-х годов он получил широкую известность благодаря участию в бурлескных и романтических комедиях. в период с 1940-х годов до середины 1960-х годов, грант активно снимался в кино, преимущественно в фильмах альфреда хичкока, говарда хоукса, сидни шелдон и стэнли донена. ряд кинокартин с участием актёра являются классикой американского кино, а десять фильмов были внесены в национальный реестр фильмов библиотеки конгресса, представляющих культурное, историческое или эстетическое значение. за свою кинокарьеру грант дважды был номинирован на кинопремию «оскар» и на пять премий «золотого глобуса», а также считается одним из самых выдающихся актёров в истории американского кинематографа и «золотого века» голливуда."
print(demo)

print("\n— Токенизация по словам:")
print(re.findall(r"[а-яёa-zA-Z]+(?:-[а-яёa-zA-Z]+)*", demo))

print("\n— BPE:")
print(sp_bpe.encode(demo, out_type=str))

print("\n— Unigram:")
print(sp_uni.encode(demo, out_type=str))


=== Пример ===
в список творческих работ англо-американского актёра кэри гранта (1904—1986) входят работы в кинематографе, в театре и на радио, которые охватывают 46 лет его актёрской карьеры (с 1920 по 1966 год). за годы кинокарьеры с 1932 по 1966 год он снялся в семидесяти двух полнометражных фильмах. начав свою актёрскую карьеру в 1920-е годы с участия в водевилях и на бродвее, в 1931 году грант перебрался в голливуд, где он начал сниматься в небольших ролях в комедийных фильмах и драмах докодексового периода. в конце 1930-х годов он получил широкую известность благодаря участию в бурлескных и романтических комедиях. в период с 1940-х годов до середины 1960-х годов, грант активно снимался в кино, преимущественно в фильмах альфреда хичкока, говарда хоукса, сидни шелдон и стэнли донена. ряд кинокартин с участием актёра являются классикой американского кино, а десять фильмов были внесены в национальный реестр фильмов библиотеки конгресса, представляющих культурное, историческое или эс